In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# --- 1. SETUP ---
MAIN_DATA_FILE = 'C:/Users/511232/Desktop/DSS/MERGING GOOGLESHEETS QUESTIONNAIRES/codes/arabic_questionnaires.xlsx'
CRITERIA_FILE = 'C:/Users/511232/Desktop/criterias.xlsx'
OUTPUT_FILE = 'availability_results_final.xlsx'

main_df = pd.read_excel(MAIN_DATA_FILE)
criteria_df = pd.read_excel(CRITERIA_FILE).dropna(subset=['Indicator_Ar'])[['Indicator_Ar', 'criteria']]

invalid_entries = ['Not applicable', 'غير متاح', 'Total']

In [ ]:
# --- 2. GENERAL AVAILABILITY PREP ---
# Convert 'Value' to numeric, forcing errors (text like 'Not applicable') to NaN
main_df['val_numeric'] = pd.to_numeric(main_df['القيمة'], errors='coerce')

# groupby(ind, country, year) and get max. This keeps the year if ANY value was numeric.
# We do NOT remove blanks yet so the timeline stays intact for the bracket check.
gen_yearly = main_df.groupby(['المؤشر', 'الدولة', 'السنة'])['val_numeric'].max().reset_index()

# Add the 'exist data' flag: 1 if the max value found is a number, else 0
gen_yearly['exist_data'] = gen_yearly['val_numeric'].notna().astype(int)

# Create Year Brackets (2010-14, 2015-19, 2020-26)
conditions = [
    (gen_yearly['السنة'] >= 2010) & (gen_yearly['السنة'] <= 2014),
    (gen_yearly['السنة'] >= 2015) & (gen_yearly['السنة'] <= 2019),
    (gen_yearly['السنة'] >= 2020) & (gen_yearly['السنة'] <= 2026)
]
choices = ['2010-2014', '2015-2019', '2020-2026']
gen_yearly['bracket'] = np.select(conditions, choices, default='Other')

# Filter for the 3 target brackets and sum the 'exist_data' flag
gen_bracket = gen_yearly[gen_yearly['bracket'] != 'Other'].groupby(['المؤشر', 'الدولة', 'bracket'])['exist_data'].sum().reset_index()

# Merge with criteria and check if each bracket is met
gen_merged = pd.merge(gen_bracket, criteria_df, left_on='المؤشر', right_on='Indicator_Ar', how='left')
gen_merged['met'] = (gen_merged['exist_data'] >= gen_merged['criteria']).astype(int)

# Final calculation: Must have met 3 brackets AND have 3 unique brackets present
final_gen = gen_merged.groupby(['المؤشر', 'الدولة']).agg(met_sum=('met', 'sum'), b_count=('bracket', 'nunique')).reset_index()
final_gen['general_availability'] = ((final_gen['met_sum'] == 3) & (final_gen['b_count'] == 3)).astype(int)

In [ ]:
# --- 3. NATIONALITY AVAILABILITY ---
# Remove blanks and inapplicables from the original df first
df_nat = main_df.dropna(subset=['المواطنة', 'val_numeric'])
df_nat = df_nat[~df_nat['المواطنة'].astype(str).isin(invalid_entries)]

# groupby(cntry, ind, year) and take max of numeric value. 
# This confirms if EITHER National or Non-National has data for that year.
nat_yearly = df_nat.groupby(['المؤشر', 'الدولة', 'السنة'])['val_numeric'].max().reset_index()
nat_yearly['exist_data'] = 1 # These are all valid because we dropped NaNs above

# Recode brackets and continue logic
nat_yearly['bracket'] = np.select([
    (nat_yearly['السنة'] >= 2010) & (nat_yearly['السنة'] <= 2014),
    (nat_yearly['السنة'] >= 2015) & (nat_yearly['السنة'] <= 2019),
    (nat_yearly['السنة'] >= 2020) & (nat_yearly['السنة'] <= 2026)
], choices, default='Other')

nat_bracket = nat_yearly[nat_yearly['bracket'] != 'Other'].groupby(['المؤشر', 'الدولة', 'bracket'])['exist_data'].sum().reset_index()
nat_merged = pd.merge(nat_bracket, criteria_df, left_on='المؤشر', right_on='Indicator_Ar', how='left')
nat_merged['met'] = (nat_merged['exist_data'] >= nat_merged['criteria']).astype(int)

final_nat = nat_merged.groupby(['المؤشر', 'الدولة']).agg(met_sum=('met', 'sum'), b_count=('bracket', 'nunique')).reset_index()
final_nat['nationality_availability'] = ((final_nat['met_sum'] == 3) & (final_nat['b_count'] == 3)).astype(int)



In [ ]:
# --- 4. SEX AVAILABILITY ---
# Remove blanks and inapplicables from original df first
df_sex = main_df.dropna(subset=['الجنس', 'val_numeric'])
df_sex = df_sex[~df_sex['الجنس'].astype(str).isin(invalid_entries)]

# groupby(cntry, ind, year) and take max. Confirms if EITHER Male or Female has data.
sex_yearly = df_sex.groupby(['المؤشر', 'الدولة', 'السنة'])['val_numeric'].max().reset_index()
sex_yearly['exist_data'] = 1

# Recode brackets and continue logic
sex_yearly['bracket'] = np.select([
    (sex_yearly['السنة'] >= 2010) & (sex_yearly['السنة'] <= 2014),
    (sex_yearly['السنة'] >= 2015) & (sex_yearly['السنة'] <= 2019),
    (sex_yearly['السنة'] >= 2020) & (sex_yearly['السنة'] <= 2026)
], choices, default='Other')

sex_bracket = sex_yearly[sex_yearly['bracket'] != 'Other'].groupby(['المؤشر', 'الدولة', 'bracket'])['exist_data'].sum().reset_index()
sex_merged = pd.merge(sex_bracket, criteria_df, left_on='المؤشر', right_on='Indicator_Ar', how='left')
sex_merged['met'] = (sex_merged['exist_data'] >= sex_merged['criteria']).astype(int)

final_sex = sex_merged.groupby(['المؤشر', 'الدولة']).agg(met_sum=('met', 'sum'), b_count=('bracket', 'nunique')).reset_index()
final_sex['sex_availability'] = ((final_sex['met_sum'] == 3) & (final_sex['b_count'] == 3)).astype(int)



In [ ]:
# --- 5. MERGE & SAVE ---
results = pd.merge(final_gen[['المؤشر', 'الدولة', 'general_availability']], 
                    final_nat[['المؤشر', 'الدولة', 'nationality_availability']], on=['المؤشر', 'الدولة'], how='left')
results = pd.merge(results, 
                    final_sex[['المؤشر', 'الدولة', 'sex_availability']], on=['المؤشر', 'الدولة'], how='left')

results.fillna(0).astype(int).to_excel(OUTPUT_FILE, index=False)
print(f"Final report saved to {OUTPUT_FILE}")